# Agentic SQL Assistant

This notebook extends the SQL Reflection experiment to build an agent that generates, executes and improves SQL queries using OpenAI and SQLite.

The project focuses on four capabilities:

1. **SQL safety:** Restrict execution to read-only queries.
2. **Error recovery:** Detect and correct SQL execution errors.
3. **Iterative reflection:** Improve queries through multiple feedback cycles.
4. **Evaluation:** Test performance against questions with known answers.

Each capability is introduced incrementally.

### Architecture

### Architecture

<p align="center">
  <img src="images/agentic_sql_assistant_workflow.png" width="425">
</p>

In [1]:
import sqlite3
import pandas as pd

## 1. SQL Safety

Before executing an LLM-generated SQL query, check that it is read-only.

Only `SELECT` queries are permitted. Additional SQLite safeguards will help prevent database modifications.

In [2]:
def validate_sql(query: str) -> bool:
    """Check whether a SQL query starts with SELECT."""
    return query.strip().upper().startswith("SELECT")

### 1.1 SQLite Authoriser

Allow database reads and approved SQL functions; reject other operations.

In [3]:
def read_only_authorizer(action, arg1, arg2, db_name, source):
    """Allow reading data and reject other database operations."""

    allowed_actions = {
        sqlite3.SQLITE_SELECT,
        sqlite3.SQLITE_READ,
        sqlite3.SQLITE_FUNCTION,
    }

    if action == sqlite3.SQLITE_FUNCTION:
        safe_functions = {
            "count", "sum", "avg", "min", "max",
            "abs", "round", "coalesce", "lower", "upper"
        }
        return (
            sqlite3.SQLITE_OK
            if (arg2 or "").lower() in safe_functions
            else sqlite3.SQLITE_DENY
        )

    return (
        sqlite3.SQLITE_OK
        if action in allowed_actions
        else sqlite3.SQLITE_DENY
    )

### 1.2 Read-Only Database Execution

Execute validated SQL using a read-only SQLite connection and the authoriser.

In [4]:
def execute_sql_readonly(query: str, db_path: str) -> pd.DataFrame:
    """Execute a SELECT query with read-only database protection."""

    if not validate_sql(query):
        raise ValueError("Only SELECT queries are allowed.")

    from pathlib import Path

    db_uri = Path(db_path).resolve().as_uri() + "?mode=ro"

    with sqlite3.connect(db_uri, uri=True) as conn:
        conn.enable_load_extension(False)
        conn.set_authorizer(read_only_authorizer)

        return pd.read_sql_query(query, conn)

## 2. SQL Safety Tests

Check basic validation, execute a sales query, and confirm that prohibited operations are blocked.

In [5]:
# Test read-only and modifying SQL statements
test_queries = [
    "SELECT * FROM transactions",
    "SELECT color, COUNT(*) FROM transactions GROUP BY color",
    "DELETE FROM transactions",
    "DROP TABLE transactions",
    "UPDATE transactions SET color = 'red'"
]

for query in test_queries:
    print(f"{validate_sql(query)}: {query}")

True: SELECT * FROM transactions
True: SELECT color, COUNT(*) FROM transactions GROUP BY color
False: DELETE FROM transactions
False: DROP TABLE transactions
False: UPDATE transactions SET color = 'red'


In [6]:
query = """
SELECT color, SUM(ABS(qty_delta) * unit_price) AS total_sales
FROM transactions
WHERE action = 'sale'
GROUP BY color
ORDER BY total_sales DESC
LIMIT 1
"""

df = execute_sql_readonly(query, "products.db")
display(df)

,color,total_sales
0,white,358315.09


In [7]:
# Verify that unsafe queries are rejected
for query in [
    "DELETE FROM transactions",
    "SELECT sqlite_version()",
    "SELECT load_extension('test_extension')",
]:
    try:
        execute_sql_readonly(query, "products.db")
        print(f"Unexpectedly allowed: {query}")
    except Exception as error:
        print(f"Blocked: {query} ({error})")

Blocked: DELETE FROM transactions (Only SELECT queries are allowed.)
Blocked: SELECT sqlite_version() (Execution failed on sql 'SELECT sqlite_version()': not authorized to use function: sqlite_version)
Blocked: SELECT load_extension('test_extension') (Execution failed on sql 'SELECT load_extension('test_extension')': not authorized to use function: load_extension)


## 2. Automatic SQL Error Recovery

When a generated SQL query fails, the assistant uses the database error message to identify and correct the problem.

The corrected query is validated and executed again, with a maximum of one correction attempt.

In [8]:
import aisuite as ai
from dotenv import load_dotenv

load_dotenv()
client = ai.Client()

In [9]:
def correct_sql(question: str, query: str, error: str, schema: str) -> str:
    """Use an LLM to correct a SQL query based on an execution error."""

    prompt = f"""
    You are a SQLite expert. Correct the SQL query using the error message.

    User question:
    {question}

    Database schema:
    {schema}

    Failed SQL:
    {query}

    Execution error:
    {error}

    Return only the corrected SQL query.
    """

    response = client.chat.completions.create(
        model="openai:gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    return response.choices[0].message.content.strip()

In [10]:
# Deliberately incorrect SQL query
question = "Which product colour has the highest total sales?"
incorrect_sql = "SELECT SUM(sales_amount) FROM transactions"

# Retrieve the database error
try:
    execute_sql_readonly(incorrect_sql, "products.db")
except Exception as e:
    error = str(e)

# Ask OpenAI to correct the query
schema = "transactions(id, product_id, product_name, brand, category, color, action, qty_delta, unit_price, notes, ts)"

corrected_sql = correct_sql(question, incorrect_sql, error, schema)

print("Corrected SQL:")
print(corrected_sql)

Corrected SQL:
SELECT color, SUM(qty_delta * unit_price) AS total_sales
FROM transactions
GROUP BY color
ORDER BY total_sales DESC
LIMIT 1;


In [11]:
df = execute_sql_readonly(corrected_sql, "products.db")
display(df)

,color,total_sales
0,blue,-150511.18


## 3. SQL Reflection with Execution Feedback

Correcting SQL execution errors does not guarantee that the query answers the user's question correctly.

Reflection allows the LLM to review the SQL query alongside its execution results, identify potential logical errors and suggest improvements.

In [12]:
def reflect_sql(question: str, query: str, result: pd.DataFrame, schema: str) -> str:
    """Improve a SQL query using its execution results."""

    prompt = f"""
    You are a SQLite expert. Review the SQL query and its results.

    User question:
    {question}

    Database schema:
    {schema}

    Important database rules:
    - Each row represents a product transaction event.
    - The action column identifies sales, restocking and other events.
    - Sales have action = 'sale' and negative qty_delta values.
    - Sales revenue is calculated as -qty_delta * unit_price.
    - Only sale events should contribute to total sales revenue.

    SQL query:
    {query}

    Execution results:
    {result.to_string(index=False)}

    Identify any logical errors and correct the SQL if necessary.
    If the query is correct, return it unchanged.

    Return only the SQL query.
    """

    response = client.chat.completions.create(
        model="openai:gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    return response.choices[0].message.content.strip()

In [13]:
# Reflect on the corrected SQL from Section 2
refined_sql = reflect_sql(
    question=question,
    query=corrected_sql,
    result=df,
    schema=schema
)

refined_sql = refined_sql.strip().removeprefix("```sql").removesuffix("```").strip()

print("Refined SQL:")
print(refined_sql)

Refined SQL:
SELECT color, SUM(-qty_delta * unit_price) AS total_sales
FROM transactions
WHERE action = 'sale'
GROUP BY color
ORDER BY total_sales DESC
LIMIT 1;


In [14]:
df_final = execute_sql_readonly(refined_sql, "products.db")
display(df_final)

,color,total_sales
0,white,358315.09
